In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np


url = "https://finance.yahoo.com/markets/crypto/all/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 1. Listas globales en PLURAL para almacenar los resultados finales
nombres = []
precios = []
variaciones_24h = []
volumenes = []

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status() 
    print("¡Conexión exitosa! Evitamos el error 403.")

    soup = BeautifulSoup(response.text, 'html.parser')
    filas = soup.select('table tbody tr')

    if not filas:
        raise ValueError("No se encontraron filas. La estructura de Yahoo Finance pudo cambiar.")

    print(f'Cacería exitosa: se encontraron {len(filas)} criptomonedas en la tabla\n')

    print("--- Analizando las primeras filas ---")
    for i, fila in enumerate(filas):
        columnas = fila.find_all('td')
        
        # Yahoo Finance tiene más de 7 columnas por fila
        if len(columnas) >= 7: 
            # 2. Variables en SINGULAR para extraer el texto de la fila actual
            nombre = columnas[1].text.strip()
            precio = columnas[3].text.strip()
            variacion = columnas[4].text.strip()  # Índice 4 es el porcentaje (Change %)
            volumen = columnas[6].text.strip()    # Índice 6 es el Volumen de 24h

            # 3. Agregamos de forma segura a las listas en PLURAL
            nombres.append(nombre)
            precios.append(precio)
            variaciones_24h.append(variacion)
            volumenes.append(volumen)

        else:
            print(f"Fila {i + 1} incompleta.")

        datos = {
            'Nombre': nombres,
            'Precios': precios,
            'Variacion_24h': variaciones_24h,
            'Volumen': volumenes
        }


except requests.exceptions.RequestException as e:
    print(f"Error de red: {e}")
except Exception as e:
    print(f"Fallo en la extracción: {e}")


df_crypto = pd.DataFrame(datos)

df_crypto

¡Conexión exitosa! Evitamos el error 403.
Cacería exitosa: se encontraron 25 criptomonedas en la tabla

--- Analizando las primeras filas ---


,Nombre,Precios,Variacion_24h,Volumen
0,Bitcoin USD,"79,828.51 -163.48 (-0.20%)",-163.48,1.603T
1,Ethereum USD,"2,509.70 -3.99 (-0.16%)",-3.99,306.416B
2,Tether USDt USD,1.00 -0.00 (-0.02%),-0.00,183.385B
3,BNB USD,749.01 -12.92 (-1.70%),-12.92,99.738B
4,XRP USD,1.41 -0.01 (-0.76%),-0.01,88.757B
5,USDC USD,1.00 +0.00 (+0.00%),+0.00,74.548B
6,Solana USD,105.44 -1.00 (-0.94%),-1.00,61.888B
7,TRON USD,0.34 +0.00 (+0.87%),+0.00,31.967B
8,Lido Staked ETH USD,"2,507.81 -1.13 (-0.05%)",-1.13,24.22B
9,Hyperliquid USD,86.69 -0.12 (-0.14%),-0.12,21.851B


In [2]:
#primero separemos el precio del porcentaje de variacion con pandas, .str.split() separa en una lista los elementos de esa columna y .str[0] toma solo el primer elemento de la lista

df_crypto['Precios'] = df_crypto['Precios'].str.split().str[0]
df_crypto['Precios'] = df_crypto['Precios'].str.replace(',', '').astype(float)
df_crypto['Variacion_24h'] = df_crypto['Variacion_24h'].astype(float)

multiplicadores = {'K': 1e3, 'M': 1e6, 'B': 1e9, 'T': 1e12}
sufijo = df_crypto['Volumen'].str[-1].str.upper()

df_crypto['Volumen'] = df_crypto['Volumen'].str.replace(r'[^\d\.]', '', regex=True).astype(float)
df_crypto['Volumen'] = df_crypto['Volumen'] * sufijo.map(multiplicadores).fillna(1)


df_crypto

,Nombre,Precios,Variacion_24h,Volumen
0,Bitcoin USD,79828.51,-163.48,1.603000e+12
1,Ethereum USD,2509.70,-3.99,3.064160e+11
2,Tether USDt USD,1.00,-0.00,1.833850e+11
3,BNB USD,749.01,-12.92,9.973800e+10
4,XRP USD,1.41,-0.01,8.875700e+10
5,USDC USD,1.00,0.00,7.454800e+10
6,Solana USD,105.44,-1.00,6.188800e+10
7,TRON USD,0.34,0.00,3.196700e+10
8,Lido Staked ETH USD,2507.81,-1.13,2.422000e+10
9,Hyperliquid USD,86.69,-0.12,2.185100e+10


In [3]:
df_crypto['Tendencia'] =np.where(df_crypto['Variacion_24h'] > 0, 'Alcista', 'Bajista')

df_crypto

,Nombre,Precios,Variacion_24h,Volumen,Tendencia
0,Bitcoin USD,79828.51,-163.48,1.603000e+12,Bajista
1,Ethereum USD,2509.70,-3.99,3.064160e+11,Bajista
2,Tether USDt USD,1.00,-0.00,1.833850e+11,Bajista
3,BNB USD,749.01,-12.92,9.973800e+10,Bajista
4,XRP USD,1.41,-0.01,8.875700e+10,Bajista
5,USDC USD,1.00,0.00,7.454800e+10,Bajista
6,Solana USD,105.44,-1.00,6.188800e+10,Bajista
7,TRON USD,0.34,0.00,3.196700e+10,Bajista
8,Lido Staked ETH USD,2507.81,-1.13,2.422000e+10,Bajista
9,Hyperliquid USD,86.69,-0.12,2.185100e+10,Bajista
